In [5]:
import pandas as pd
import numpy as np

excel_path=r'E:\Github\Block A 2\Task 6\dataset(asli).xlsx'
df = pd.read_excel(excel_path)
print(df.head())

                                                text  Anger  Fear  Happiness  \
0  کرونا رو شکست میدهیم؟\nمرحله بعد چه گوهی میخوا...      4     3          1   
1  اگر در چند ماه اخیر تصمیم داشته اید وارد بورس ...      2     4          1   
2  یکی از پدرسوختگی های #برانداز اینه که ظاهرا ژس...      5     3          2   
3                              یکی از دوستای دبستانم      1     0          0   
4  @username اینقدر گرفتار مسایل میشی که تخصص از ...      2     1          0   

   Hatred  Sadness  Wonder  
0       3        3       4  
1       2        4       2  
2       4        5       2  
3       1        0       0  
4       1        0       4  


In [7]:
mapping = {'Wonder': 'surprize', 
            'Hatred': 'disgust',
            'Happiness': 'happiness',
            'Sadness': 'sadness',
            'Fear': 'fear',
            'Anger': 'anger',}
df= df.rename(columns=mapping)
print(df.head())

                                                text  anger  fear  happiness  \
0  کرونا رو شکست میدهیم؟\nمرحله بعد چه گوهی میخوا...      4     3          1   
1  اگر در چند ماه اخیر تصمیم داشته اید وارد بورس ...      2     4          1   
2  یکی از پدرسوختگی های #برانداز اینه که ظاهرا ژس...      5     3          2   
3                              یکی از دوستای دبستانم      1     0          0   
4  @username اینقدر گرفتار مسایل میشی که تخصص از ...      2     1          0   

   disgust  sadness  surprize  
0        3        3         4  
1        2        4         2  
2        4        5         2  
3        1        0         0  
4        1        0         4  


In [8]:
EMOTIONS = ["anger","fear","happiness","disgust","sadness","surprize"]
def to_single_label(row):
    scores = row[EMOTIONS].values
    if np.all(scores==0): 
        return "neutral"
    return EMOTIONS[np.argmax(scores)]

df["label"] = df.apply(to_single_label, axis=1)

In [10]:
def to_single_label_thresh(row, thr=1):
    scores = row[EMOTIONS].values
    if scores.max() <= thr:
        return "neutral"
    return EMOTIONS[scores.argmax()]
df["label"] = df.apply(to_single_label_thresh, axis=1)

In [9]:
df.head()

,text,anger,fear,happiness,disgust,sadness,surprize,label
0,کرونا رو شکست میدهیم؟\nمرحله بعد چه گوهی میخوا...,4,3,1,3,3,4,anger
1,اگر در چند ماه اخیر تصمیم داشته اید وارد بورس ...,2,4,1,2,4,2,fear
2,یکی از پدرسوختگی های #برانداز اینه که ظاهرا ژس...,5,3,2,4,5,2,anger
3,یکی از دوستای دبستانم,1,0,0,1,0,0,anger
4,@username اینقدر گرفتار مسایل میشی که تخصص از ...,2,1,0,1,0,4,surprize


In [11]:
import pandas as pd
import re

df = pd.DataFrame(df)

# Function to clean text
def clean_text(text):
    # 1. Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # 2. Remove English words (sequences of a-z or A-Z)
    text = re.sub(r'[a-zA-Z]+', '', text)
    # 3. Remove single English letters
    text = re.sub(r'\b[a-zA-Z]\b', '', text)
    # 4. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
df["text"] = df["text"].apply(clean_text)

print(df)

                                                    text  anger  fear  \
0      کرونا رو شکست میدهیم؟ مرحله بعد چه گوهی میخوای...      4     3   
1      اگر در چند ماه اخیر تصمیم داشته اید وارد بورس ...      2     4   
2      یکی از پدرسوختگی های #برانداز اینه که ظاهرا ژس...      5     3   
3                                  یکی از دوستای دبستانم      1     0   
4      اینقدر گرفتار مسایل میشی که تخصص از دستت درد م...      2     1   
...                                                  ...    ...   ...   
29995  اتفاقا جامعه دوپاره نیست،اصلاحات چی های متوهم ...      2     1   
29996  چرا مخالف #تحریم‌ ها بودیم و هستیم و چرا با #ب...      2     0   
29997  اصلا اینطور نیست. ببین توی ایران سرمایه سرگردا...      0     0   
29998  نفس نفسمان شده کنکور شب و روزمان شده تلاش برای...      1     1   
29999    تا کرونا تموم بشه که ان شاالله زودتر تموم بشه😇🤗      0     1   

       happiness  disgust  sadness  surprize     label  
0              1        3        3         4     anger  
1        

In [ ]:
import stanza
import pandas as pd


stanza.download("fa")  # run only once
nlp = stanza.Pipeline("fa", processors="tokenize,pos", verbose=False)


# 3. Function: extract raw POS tags
def get_pos_tags(sentence: str):
    doc = nlp(str(sentence))
    return [word.upos for sent in doc.sentences for word in sent.words]

# 4. Build UPOS vocabulary
all_tags = set()
for sentence in df["text"]:   # assumes you have a column named "Sentence"
    all_tags.update(get_pos_tags(sentence))
upos_map = {tag: idx for idx, tag in enumerate(sorted(all_tags))}

# 5. Function: encode POS tags
def get_pos_tags_encoded(sentence: str):
    tags = get_pos_tags(sentence)
    return [upos_map[tag] for tag in tags]

# 6. Add new columns
df["POS_Tags"] = df["text"].apply(get_pos_tags)
df["POS_Encoded"] = df["text"].apply(get_pos_tags_encoded)

# 7. Save back into the same Excel file (overwrite with updated version)
df.to_excel(excel_path, index=False)

print(f"✅ POS tags added and saved back into {excel_path}")


✅ POS tags added and saved back into E:\Github\Block A 2\Task 6\dataset(asli).xlsx


In [15]:
import pandas as pd
import numpy as np

excel_path=r'E:\Github\Block A 2\Task 6\dataset(asli).xlsx'
df = pd.read_excel(excel_path)
print(df.head())

                                                text  anger  fear  happiness  \
0  کرونا رو شکست میدهیم؟ مرحله بعد چه گوهی میخوای...      4     3          1   
1  اگر در چند ماه اخیر تصمیم داشته اید وارد بورس ...      2     4          1   
2  یکی از پدرسوختگی های #برانداز اینه که ظاهرا ژس...      5     3          2   
3                              یکی از دوستای دبستانم      1     0          0   
4  اینقدر گرفتار مسایل میشی که تخصص از دستت درد م...      2     1          0   

   disgust  sadness  surprize     label  \
0        3        3         4     anger   
1        2        4         2      fear   
2        4        5         2     anger   
3        1        0         0   neutral   
4        1        0         4  surprize   

                                            POS_Tags  \
0  ['PROPN', 'ADP', 'NOUN', 'VERB', 'PUNCT', 'NOU...   
1  ['SCONJ', 'ADP', 'DET', 'NOUN', 'ADJ', 'NOUN',...   
2  ['NOUN', 'ADP', 'NOUN', 'NOUN', 'ADJ', 'PRON',...   
3            ['NOUN', 'ADP', 'NOUN',

In [18]:
from sklearn.preprocessing import LabelEncoder
label_encoder=LabelEncoder()
df["Label_encoded"]=label_encoder.fit_transform(df["label"])
print(df.head())

                                                text  anger  fear  happiness  \
0  کرونا رو شکست میدهیم؟ مرحله بعد چه گوهی میخوای...      4     3          1   
1  اگر در چند ماه اخیر تصمیم داشته اید وارد بورس ...      2     4          1   
2  یکی از پدرسوختگی های #برانداز اینه که ظاهرا ژس...      5     3          2   
3                              یکی از دوستای دبستانم      1     0          0   
4  اینقدر گرفتار مسایل میشی که تخصص از دستت درد م...      2     1          0   

   disgust  sadness  surprize     label  \
0        3        3         4     anger   
1        2        4         2      fear   
2        4        5         2     anger   
3        1        0         0   neutral   
4        1        0         4  surprize   

                                            POS_Tags  \
0  ['PROPN', 'ADP', 'NOUN', 'VERB', 'PUNCT', 'NOU...   
1  ['SCONJ', 'ADP', 'DET', 'NOUN', 'ADJ', 'NOUN',...   
2  ['NOUN', 'ADP', 'NOUN', 'NOUN', 'ADJ', 'PRON',...   
3            ['NOUN', 'ADP', 'NOUN',

In [22]:
df["text"] = df["text"].fillna("").astype(str)

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(analyzer="char",max_features=10000,ngram_range=(3,5))
X_text = tfidf.fit_transform(df["text"])


In [26]:
df['POS_Encoded'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 30000 entries, 0 to 29999
Series name: POS_Encoded
Non-Null Count  Dtype 
--------------  ----- 
30000 non-null  object
dtypes: object(1)
memory usage: 234.5+ KB


In [28]:
df["POS_doc"] = df["POS_Tags"].apply(lambda tags: " ".join(tags))

from sklearn.feature_extraction.text import TfidfVectorizer

# simple: unigrams of POS tags
pos_vec = TfidfVectorizer(analyzer="word", vocabulary=sorted(upos_map.keys()))
X_pos = pos_vec.fit_transform(df["POS_doc"])

e:\miniconda\envs\NLP\lib\site-packages\sklearn\feature_extraction\text.py:1369: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


In [30]:
df.columns

Index(['text', 'anger', 'fear', 'happiness', 'disgust', 'sadness', 'surprize',
       'label', 'POS_Tags', 'POS_Encoded', 'Label_encoded', 'POS_doc'],
      dtype='object')

In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
X = hstack([X_text, X_pos])
y = df["Label_encoded"]

# -----------------------------
# Step 5: Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
clf = LogisticRegression(max_iter=1000, solver="lbfgs", multi_class="multinomial")
clf.fit(X_train, y_train)

# -----------------------------
# Step 7: Evaluation
# -----------------------------
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


e:\miniconda\envs\NLP\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

       anger       0.23      0.23      0.23      1084
     disgust       0.17      0.03      0.06       635
        fear       0.12      0.02      0.04       577
   happiness       0.21      0.01      0.02       508
     neutral       0.35      0.78      0.49      2017
     sadness       0.16      0.04      0.07       768
    surprize       0.07      0.00      0.00       411

    accuracy                           0.32      6000
   macro avg       0.19      0.16      0.13      6000
weighted avg       0.23      0.32      0.23      6000



Iteration 2

In [35]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",   # default in new versions
    n_jobs=-1,
    class_weight="balanced",
     multi_class="multinomial" # 🔑 balances minority vs majority classes
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

e:\miniconda\envs\NLP\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

       anger       0.22      0.16      0.18      1084
     disgust       0.12      0.16      0.14       635
        fear       0.12      0.18      0.14       577
   happiness       0.13      0.22      0.17       508
     neutral       0.39      0.18      0.25      2017
     sadness       0.13      0.14      0.14       768
    surprize       0.08      0.16      0.10       411

    accuracy                           0.17      6000
   macro avg       0.17      0.17      0.16      6000
weighted avg       0.23      0.17      0.18      6000



Iteration 3

In [36]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
print("Custom class weights:", class_weights)

clf = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    n_jobs=-1,
    class_weight=class_weights,
    multi_class="multinomial"
)

Custom class weights: {np.int64(0): np.float64(0.7658189476371294), np.int64(1): np.float64(1.2977181788688223), np.int64(2): np.float64(1.4733869482472834), np.int64(3): np.float64(1.680672268907563), np.int64(4): np.float64(0.43598314198517657), np.int64(5): np.float64(1.1717605702568108), np.int64(6): np.float64(1.9887305270135898)}


In [37]:
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

e:\miniconda\envs\NLP\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

       anger       0.22      0.16      0.18      1084
     disgust       0.12      0.16      0.14       635
        fear       0.12      0.18      0.14       577
   happiness       0.13      0.22      0.17       508
     neutral       0.39      0.18      0.25      2017
     sadness       0.13      0.14      0.14       768
    surprize       0.08      0.16      0.10       411

    accuracy                           0.17      6000
   macro avg       0.17      0.17      0.16      6000
weighted avg       0.23      0.17      0.18      6000



different

In [42]:
import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.pipeline import FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin

# -------------------------
# 0) Persian normalization
# -------------------------
def fa_normalize(s: str) -> str:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = (s.replace("\u064A", "\u06CC")   # Arabic Yeh -> Farsi Yeh
           .replace("\u0643", "\u06A9")   # Arabic Kaf -> Keheh
           .replace("\u0640", ""))        # Tatweel
    s = re.sub(r"[\u200C\u200D\u200E\u200F]", " ", s)   # zero-widths -> space
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text"] = df["text"].fillna("").map(fa_normalize)

# -------------------------
# 1) Labels -> integers
# -------------------------
CANDIDATES = ["label", "Label", "emotion", "Emotion", "target", "y"]
label_col = next((c for c in CANDIDATES if c in df.columns), None)
assert label_col is not None, f"Label column not found. Have: {list(df.columns)}"

le = LabelEncoder()
y = le.fit_transform(df[label_col].astype(str))

# -------------------------
# 2) Text feature extractors
# -------------------------
# char n-grams (great for Persian morphology/orthography)
char_tfidf = TfidfVectorizer(analyzer="char", ngram_range=(3,5), max_features=20000, min_df=3)

# word n-grams (captures lexical cues; keep modest size)
word_tfidf = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=30000, min_df=2)

X_char = char_tfidf.fit_transform(df["text"])
X_word = word_tfidf.fit_transform(df["text"])

# -------------------------
# 3) POS TF-IDF (use your existing UPOS tags)
#    You already have df["POS_Tags"] as list[str]
# -------------------------
assert "POS_Tags" in df.columns, "Expected df['POS_Tags'] with UPOS tags"
df["POS_doc"] = df["POS_Tags"].apply(lambda tags: " ".join(tags))
pos_tfidf = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2)
X_pos = pos_tfidf.fit_transform(df["POS_doc"])

# Combine: [char-grams | word-grams | POS-ngrams]
from scipy.sparse import csr_matrix
X = hstack([X_char, X_word, X_pos], format="csr")

# -------------------------
# 4) Train/Test split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------
# 5) Class weights: try balanced + smoothed custom
# -------------------------
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
base_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
# Exponent smoothing: alpha in {0.5, 0.75, 1.0}
def weights_with_alpha(alpha):
    w = np.power(base_weights, alpha)
    w = w * (len(w) / w.sum())  # normalize roughly
    return dict(zip(classes, w))

weight_sets = {
    "none": None,
    "balanced": "balanced",
    "alpha_0.5": weights_with_alpha(0.5),
    "alpha_0.75": weights_with_alpha(0.75),
    "alpha_1.0": weights_with_alpha(1.0),
}

# -------------------------
# 6) Small hyperparam search (macro-F1)
# -------------------------
from sklearn.model_selection import cross_val_score

c_vals = [0.25, 0.5, 1.0, 2.0]
solvers = [("lbfgs","l2"), ("saga","l2"), ("saga","l1")]  # l1 needs saga
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best = {"cv_f1": -1, "clf": None, "params": None}
for cw_name, cw in weight_sets.items():
    for C in c_vals:
        for solver, penalty in solvers:
            if penalty == "l1" and solver != "saga":
                continue
            clf = LogisticRegression(
                C=C,
                max_iter=2000,
                solver=solver,
                penalty=penalty,
                n_jobs=-1,
                class_weight=cw
            )
            scores = cross_val_score(clf, X_train, y_train, scoring="f1_macro", cv=cv, n_jobs=-1)
            m = scores.mean()
            # print(f"[{cw_name}] solver={solver} pen={penalty} C={C} -> macroF1={m:.4f}")
            if m > best["cv_f1"]:
                best = {"cv_f1": m, "clf": clf, "params": (cw_name, solver, penalty, C)}

print("Best CV macro-F1:", round(best["cv_f1"],4), "with params:", best["params"])

# -------------------------
# 7) Fit best and evaluate on hold-out
# -------------------------
best["clf"].fit(X_train, y_train)
y_pred = best["clf"].predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))

# Confusion matrix (optional: inspect most confused pairs)
cm = confusion_matrix(y_test, y_pred, labels=classes)
print("Confusion matrix shape:", cm.shape)


Best CV macro-F1: 0.172 with params: ('alpha_0.75', 'saga', 'l1', 1.0)
              precision    recall  f1-score   support

       anger       0.24      0.22      0.23      1112
     disgust       0.13      0.10      0.11       655
        fear       0.14      0.15      0.14       581
   happiness       0.14      0.14      0.14       510
     neutral       0.39      0.47      0.43      1976
     sadness       0.16      0.11      0.13       739
    surprize       0.09      0.08      0.09       427

    accuracy                           0.25      6000
   macro avg       0.18      0.18      0.18      6000
weighted avg       0.24      0.25      0.24      6000

Confusion matrix shape: (7, 7)


In [40]:
df.head()

,text,anger,fear,happiness,disgust,sadness,surprize,label,POS_Tags,POS_Encoded,Label_encoded,POS_doc
0,کرونا رو شکست میدهیم؟ مرحله بعد چه گوهی میخوای...,4,3,1,3,3,4,anger,"['PROPN', 'ADP', 'NOUN', 'VERB', 'PUNCT', 'NOU...","[11, 1, 7, 14, 12, 7, 0, 5, 7, 14, 14, 12, 12]",0,"[ ' P R O P N ' , ' A D P ' , ' N O U N ' ..."
1,اگر در چند ماه اخیر تصمیم داشته اید وارد بورس ...,2,4,1,2,4,2,fear,"['SCONJ', 'ADP', 'DET', 'NOUN', 'ADJ', 'NOUN',...","[13, 1, 5, 7, 0, 7, 14, 3, 0, 7, 14, 4, 1, 5, ...",2,"[ ' S C O N J ' , ' A D P ' , ' D E T ' , ..."
2,یکی از پدرسوختگی های #برانداز اینه که ظاهرا ژس...,5,3,2,4,5,2,anger,"['NOUN', 'ADP', 'NOUN', 'NOUN', 'ADJ', 'PRON',...","[7, 1, 7, 7, 0, 10, 13, 2, 7, 7, 14, 12, 4, 1,...",0,"[ ' N O U N ' , ' A D P ' , ' N O U N ' , ..."
3,یکی از دوستای دبستانم,1,0,0,1,0,0,neutral,"['NOUN', 'ADP', 'NOUN', 'NOUN', 'PRON']","[7, 1, 7, 7, 10]",4,"[ ' N O U N ' , ' A D P ' , ' N O U N ' , ..."
4,اینقدر گرفتار مسایل میشی که تخصص از دستت درد م...,2,1,0,1,0,4,surprize,"['ADV', 'ADJ', 'NOUN', 'VERB', 'SCONJ', 'NOUN'...","[2, 0, 7, 14, 13, 7, 1, 7, 10, 7, 14, 4, 1, 10...",6,"[ ' A D V ' , ' A D J ' , ' N O U N ' , ..."


In [41]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1) Ensure POS_Tags is a list of strings per row
def as_list_of_str(x):
    if isinstance(x, list):
        return [str(t) for t in x if t is not None]
    if pd.isna(x):
        return []
    # if it’s a stringified list, try to parse; else treat as single token
    try:
        import ast
        parsed = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(parsed, list):
            return [str(t) for t in parsed if t is not None]
    except Exception:
        pass
    return [str(x)]

if "POS_Tags" not in df.columns:
    raise KeyError("df['POS_Tags'] not found. Make sure you created it with Stanza.")

df["POS_Tags"] = df["POS_Tags"].apply(as_list_of_str)
df["POS_doc"] = df["POS_Tags"].apply(lambda tags: " ".join(tags))  # may be ""

# 2) Build explicit vocabulary from your corpus (set of unique POS tags)
pos_vocab = sorted({t for tags in df["POS_Tags"] for t in tags})
if len(pos_vocab) == 0:
    raise ValueError("POS vocabulary is empty. Are POS tags being extracted correctly?")

# 3) Vectorize POS “documents” with the fixed vocabulary
#    - lowercase=False keeps tags like NOUN/VERB as-is
#    - token_pattern matches any non-space token
pos_tfidf = TfidfVectorizer(
    analyzer="word",
    lowercase=False,
    vocabulary=pos_vocab,
    token_pattern=r"[^ ]+",
    ngram_range=(1, 2),  # keep (1,2); you can try (1,1) if very sparse
)
X_pos = pos_tfidf.fit_transform(df["POS_doc"])  # will not error even if some docs are empty

print("X_pos shape:", X_pos.shape)
print("Sample of POS vocab:", pos_vocab[:10])


X_pos shape: (30000, 15)
Sample of POS vocab: ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART']


Naive

In [44]:

from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, f1_score

# -------------------------
# 0) Persian normalization (lightweight, safe)
# -------------------------
def fa_normalize(s: str) -> str:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = (s.replace("\u064A", "\u06CC")   # Arabic Yeh -> Farsi Yeh
           .replace("\u0643", "\u06A9")   # Arabic Kaf -> Keheh
           .replace("\u0640", ""))        # Tatweel
    s = re.sub(r"[\u200C\u200D\u200E\u200F]", " ", s)   # zero-widths -> space
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text"] = df["text"].fillna("").map(fa_normalize)

# -------------------------
# 1) Labels -> integers
# -------------------------
CANDIDATES = ["label", "Label", "emotion", "Emotion", "target", "y"]
label_col = next((c for c in CANDIDATES if c in df.columns), None)
assert label_col is not None, f"Label column not found. Have: {list(df.columns)}"

le = LabelEncoder()
y = le.fit_transform(df[label_col].astype(str))

# -------------------------
# 2) Text features
# -------------------------
char_tfidf = TfidfVectorizer(analyzer="char", ngram_range=(3,5), max_features=20000, min_df=3)
word_tfidf = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=30000, min_df=2)

X_char = char_tfidf.fit_transform(df["text"])
X_word = word_tfidf.fit_transform(df["text"])

# -------------------------
# 3) POS TF-IDF features (uses your existing df["POS_Tags"])
# -------------------------
def as_list_of_str(x):
    if isinstance(x, list):
        return [str(t) for t in x if t is not None]
    if pd.isna(x):
        return []
    try:
        import ast
        parsed = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(parsed, list):
            return [str(t) for t in parsed if t is not None]
    except Exception:
        pass
    return [str(x)]

assert "POS_Tags" in df.columns, "Expected df['POS_Tags'] with UPOS tags"

df["POS_Tags"] = df["POS_Tags"].apply(as_list_of_str)
df["POS_doc"]  = df["POS_Tags"].apply(lambda tags: " ".join(tags))

pos_vocab = sorted({t for tags in df["POS_Tags"] for t in tags})
if len(pos_vocab) == 0:
    raise ValueError("POS vocabulary is empty. Check your POS extraction.")
pos_tfidf = TfidfVectorizer(
    analyzer="word",
    lowercase=False,     # keep NOUN/VERB as-is
    vocabulary=pos_vocab,
    token_pattern=r"[^ ]+",
    ngram_range=(1,2),
)
X_pos = pos_tfidf.fit_transform(df["POS_doc"])

# -------------------------
# 4) Combine sparse features
# -------------------------
X = hstack([X_char, X_word, X_pos], format="csr")

# -------------------------
# 5) Split
# -------------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------
# 6) Small alpha search for ComplementNB and MultinomialNB
# -------------------------
alphas = [0.1, 0.3, 0.5, 1.0, 2.0]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_macro_f1_nb(model_cls, X, y, alphas):
    best = {"alpha": None, "score": -1}
    for a in alphas:
        clf = model_cls(alpha=a)
        # ComplementNB/MultinomialNB don't have class_weight; this CV uses macro-F1
        scores = []
        for tr_idx, va_idx in cv.split(X, y):
            clf.fit(X[tr_idx], y[tr_idx])
            pred = clf.predict(X[va_idx])
            scores.append(f1_score(y[va_idx], pred, average="macro"))
        m = float(np.mean(scores))
        if m > best["score"]:
            best = {"alpha": a, "score": m}
    return best

best_cnb = cv_macro_f1_nb(ComplementNB, X_tr, y_tr, alphas)
best_mnb = cv_macro_f1_nb(MultinomialNB, X_tr, y_tr, alphas)

print(f"Best ComplementNB: alpha={best_cnb['alpha']}  CV macro-F1={best_cnb['score']:.4f}")
print(f"Best MultinomialNB: alpha={best_mnb['alpha']}  CV macro-F1={best_mnb['score']:.4f}")

# -------------------------
# 7) Fit best model on train and evaluate on test
# -------------------------
# Prefer ComplementNB under imbalance; fall back to MNB if it wins
use_cnb = best_cnb["score"] >= best_mnb["score"]
if use_cnb:
    nb = ComplementNB(alpha=best_cnb["alpha"])
    model_name = "ComplementNB"
else:
    nb = MultinomialNB(alpha=best_mnb["alpha"])
    model_name = "MultinomialNB"

nb.fit(X_tr, y_tr)
y_pred = nb.predict(X_te)

print(f"\n=== {model_name} on holdout ===")
print(classification_report(y_te, y_pred, target_names=le.classes_))


Best ComplementNB: alpha=0.3  CV macro-F1=0.1730
Best MultinomialNB: alpha=0.1  CV macro-F1=0.1580

=== ComplementNB on holdout ===
              precision    recall  f1-score   support

       anger       0.24      0.27      0.26      1112
     disgust       0.13      0.11      0.12       655
        fear       0.15      0.18      0.16       581
   happiness       0.12      0.10      0.11       510
     neutral       0.38      0.41      0.39      1976
     sadness       0.15      0.11      0.13       739
    surprize       0.10      0.07      0.08       427

    accuracy                           0.24      6000
   macro avg       0.18      0.18      0.18      6000
weighted avg       0.23      0.24      0.24      6000



SVM

In [46]:
import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report

# -------------------------
# 1) Light Persian normalization
# -------------------------
def fa_normalize(s: str) -> str:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = (s.replace("\u064A", "\u06CC")   # Arabic Yeh -> Farsi Yeh
           .replace("\u0643", "\u06A9")   # Arabic Kaf -> Keheh
           .replace("\u0640", ""))        # Tatweel
    # Replace zero-width chars with space, collapse whitespace
    s = re.sub(r"[\u200C\u200D\u200E\u200F]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text"] = df["text"].fillna("").map(fa_normalize)

# -------------------------
# 2) Labels -> integers (works with Persian or English label strings)
# -------------------------
CANDIDATES = ["label", "Label", "emotion", "Emotion", "target", "y"]
label_col = next((c for c in CANDIDATES if c in df.columns), None)
assert label_col is not None, f"Label column not found. Have: {list(df.columns)}"

le = LabelEncoder()
y = le.fit_transform(df[label_col].astype(str))

# -------------------------
# 3) Text TF-IDF (two complementary views)
#    - char n-grams: robust to morphology, typos, ZWNJ
#    - word n-grams: lexical cues
# -------------------------
char_vec = TfidfVectorizer(analyzer="char", ngram_range=(3,5), max_features=20000, min_df=3)
word_vec = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=30000, min_df=2)

X_char = char_vec.fit_transform(df["text"])
X_word = word_vec.fit_transform(df["text"])

# -------------------------
# 4) POS TF-IDF (from your df["POS_Tags"] lists)
#    Build a stable vocabulary to avoid "empty vocabulary" issues.
# -------------------------
def as_list_of_str(x):
    if isinstance(x, list):
        return [str(t) for t in x if t is not None]
    if pd.isna(x):
        return []
    try:
        import ast
        parsed = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(parsed, list):
            return [str(t) for t in parsed if t is not None]
    except Exception:
        pass
    return [str(x)]

assert "POS_Tags" in df.columns, "Expected df['POS_Tags'] with UPOS tags from Stanza."
df["POS_Tags"] = df["POS_Tags"].apply(as_list_of_str)
df["POS_doc"]  = df["POS_Tags"].apply(lambda tags: " ".join(tags))

pos_vocab = sorted({t for tags in df["POS_Tags"] for t in tags})
if len(pos_vocab) == 0:
    raise ValueError("POS vocabulary is empty. Check POS extraction.")

pos_vec = TfidfVectorizer(
    analyzer="word",
    lowercase=False,     # keep NOUN/VERB etc.
    vocabulary=pos_vocab,
    token_pattern=r"[^ ]+",
    ngram_range=(1,2),   # try (1,1) if your corpus is small
)
X_pos = pos_vec.fit_transform(df["POS_doc"])

# -------------------------
# 5) Combine all features (sparse-safe)
# -------------------------
X = hstack([X_char, X_word, X_pos], format="csr")

# -------------------------
# 6) Split (stratify preserves class ratios)
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------
# 7) LinearSVC + small, readable hyperparameter search
#    - C controls regularization (smaller C = stronger regularization)
#    - class_weight='balanced' helps minority classes
# -------------------------
param_grid = {
    "C": [0.25, 0.5, 1.0, 2.0],
    "class_weight": ["balanced", None],
    # "loss": ["squared_hinge"],  # default; keep simple
}
svc = LinearSVC(max_iter=5000)  # no probability, but strong for text

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    scoring="f1_macro",     # macro-F1 is the right target here
    cv=cv,
    n_jobs=-1,
    verbose=0
)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV macro-F1:", round(grid.best_score_, 4))

# -------------------------
# 8) Fit best model and evaluate on holdout
# -------------------------
best_svc = grid.best_estimator_
best_svc.fit(X_train, y_train)
y_pred = best_svc.predict(X_test)

print("\n=== LinearSVC report (holdout) ===")
print(classification_report(y_test, y_pred, target_names=le.classes_))


Best params: {'C': 0.5, 'class_weight': 'balanced'}
Best CV macro-F1: 0.1621

=== LinearSVC report (holdout) ===
              precision    recall  f1-score   support

       anger       0.21      0.20      0.21      1112
     disgust       0.12      0.13      0.12       655
        fear       0.11      0.12      0.11       581
   happiness       0.12      0.15      0.13       510
     neutral       0.37      0.34      0.35      1976
     sadness       0.14      0.14      0.14       739
    surprize       0.09      0.10      0.09       427

    accuracy                           0.21      6000
   macro avg       0.17      0.17      0.17      6000
weighted avg       0.22      0.21      0.21      6000



In [47]:
def make_class_weights(y, alpha=1.0):
    classes, counts = np.unique(y, return_counts=True)
    freq = counts / counts.sum()
    weights = (1.0 / np.clip(freq, 1e-12, None)) ** alpha
    # Optional normalization (not required, but keeps magnitudes reasonable)
    weights = weights * (len(classes) / weights.sum())
    return dict(zip(classes, weights))

cw_alpha05  = make_class_weights(y_train, alpha=0.5)
cw_alpha075 = make_class_weights(y_train, alpha=0.75)
cw_alpha10  = make_class_weights(y_train, alpha=1.0)

print("Manual class weights (alpha=1.0):", cw_alpha10)

param_grid = {
    "C": [0.25, 0.5, 1.0, 2.0],
    "class_weight": [cw_alpha05, cw_alpha075, cw_alpha10],  # manual weights
}

svc = LinearSVC(max_iter=5000)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=0
)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV macro-F1:", round(grid.best_score_, 4))

# -------------------------
# 9) Fit best model and evaluate on holdout
# -------------------------
best_svc = grid.best_estimator_
best_svc.fit(X_train, y_train)
y_pred = best_svc.predict(X_test)

print("\n=== LinearSVC (manual weights) on holdout ===")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Manual class weights (alpha=1.0): {np.int64(0): np.float64(0.6103830790458531), np.int64(1): np.float64(1.0356957737128147), np.int64(2): np.float64(1.1690031505273355), np.int64(3): np.float64(1.3324800386040236), np.int64(4): np.float64(0.34352869306451617), np.int64(5): np.float64(0.9189828489593909), np.int64(6): np.float64(1.5899264160860658)}
Best params: {'C': 0.25, 'class_weight': {np.int64(0): np.float64(0.702106334736561), np.int64(1): np.float64(1.0438189578670978), np.int64(2): np.float64(1.1430435791578226), np.int64(3): np.float64(1.2609462111003624), np.int64(4): np.float64(0.45621891789313385), np.int64(5): np.float64(0.9542926565367469), np.int64(6): np.float64(1.4395733427082753)}}
Best CV macro-F1: 0.1641

=== LinearSVC (manual weights) on holdout ===
              precision    recall  f1-score   support

       anger       0.22      0.21      0.22      1112
     disgust       0.13      0.12      0.12       655
        fear       0.11      0.11      0.11       581
  